# Revolving-Credit Default: Source Exploration

This notebook starts from warehouse dimensions and facts. It asks whether their grain, history, and behaviour support a 90-day existing-account default early-warning model.

The model is for portfolio monitoring and case prioritisation, not automated approval or adverse action. Protected characteristics are excluded.

In [ ]:
from pathlib import Path

import yaml
import modin.pandas as pd
import snowflake.snowpark.modin.plugin
import matplotlib.pyplot as plt
from IPython.display import display
from snowflake.snowpark.context import get_active_session

session = get_active_session()
config_path = Path("../project.yaml")
if not config_path.is_file():
    config_path = Path("project.yaml")
config = yaml.safe_load(config_path.read_text())

database_name = config["snowflake"]["database"]
raw_schema = config["snowflake"]["schemas"]["raw"]
session.use_role(config["snowflake"]["roles"]["developer"])
session.use_warehouse(config["snowflake"]["warehouse"])
session.use_database(database_name)
session.use_schema(raw_schema)

## 1. What source data exists?

Before proposing features, inspect the documented warehouse contract. The expected progression is customer and facility dimensions, dated financial estimates, daily servicing state, and atomic events.

In [ ]:
%%sql -r source_contract
SELECT table_name, column_name, data_type, comment
FROM CRISK_DEMO_DB.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'RAW'
ORDER BY table_name, ordinal_position

In [ ]:
source_tables = [
    "DIM_CUSTOMER", "DIM_CREDIT_ACCOUNT", "FACT_CUSTOMER_FINANCIAL_SNAPSHOT",
    "FACT_ACCOUNT_DAILY_SNAPSHOT", "FACT_PAYMENT", "FACT_CUSTOMER_CONTACT",
    "FACT_ACCOUNT_EVENT", "FACT_DEFAULT_EVENT", "ACCOUNT_OBSERVATION",
]
source_counts = []
for table_name in source_tables:
    frame = pd.read_snowflake(f"{database_name}.{raw_schema}.{table_name}")
    source_counts.append({"TABLE_NAME": table_name, "RECORD_COUNT": len(frame)})
display(pd.DataFrame(source_counts))

The dimensions should establish who and what is observed without exposing a prepared model row. Next, check their keys, relationship, and operational categories.

In [ ]:
customers = pd.read_snowflake(f"{database_name}.{raw_schema}.DIM_CUSTOMER")
accounts = pd.read_snowflake(f"{database_name}.{raw_schema}.DIM_CREDIT_ACCOUNT")

dimension_checks = pd.DataFrame({
    "CHECK": ["customer key unique", "account key unique", "all accounts find a customer"],
    "VALUE": [
        len(customers) == customers["CUSTOMER_ID"].nunique(),
        len(accounts) == accounts["ACCOUNT_ID"].nunique(),
        len(accounts.merge(customers[["CUSTOMER_ID"]], on="CUSTOMER_ID", how="inner")) == len(accounts),
    ],
})
display(dimension_checks)
display(accounts.groupby(["PRODUCT_CODE", "ORIGINATION_CHANNEL"]).size().reset_index(name="ACCOUNT_COUNT"))

The dimensions establish the account-to-customer relationship. Before analysing behaviour, check every fact at its declared grain, measure missingness, and confirm that each fact key resolves to its parent dimension.

In [ ]:
fact_contracts = {
    "FACT_CUSTOMER_FINANCIAL_SNAPSHOT": (["CUSTOMER_ID", "SNAPSHOT_DATE"], "CUSTOMER_ID", customers),
    "FACT_ACCOUNT_DAILY_SNAPSHOT": (["ACCOUNT_ID", "SNAPSHOT_DATE"], "ACCOUNT_ID", accounts),
    "FACT_PAYMENT": (["PAYMENT_ID"], "ACCOUNT_ID", accounts),
    "FACT_CUSTOMER_CONTACT": (["CONTACT_ID"], "ACCOUNT_ID", accounts),
    "FACT_ACCOUNT_EVENT": (["EVENT_ID"], "ACCOUNT_ID", accounts),
    "FACT_DEFAULT_EVENT": (["ACCOUNT_ID"], "ACCOUNT_ID", accounts),
    "ACCOUNT_OBSERVATION": (["ACCOUNT_ID", "OBSERVATION_DATE"], "ACCOUNT_ID", accounts),
}
fact_checks = []
missingness = []
for table_name, (grain_columns, parent_key, parent_frame) in fact_contracts.items():
    fact = pd.read_snowflake(f"{database_name}.{raw_schema}.{table_name}")
    duplicate_count = int(fact.duplicated(subset=grain_columns).sum())
    orphan_count = len(fact[~fact[parent_key].isin(parent_frame[parent_key])])
    fact_checks.append({
        "TABLE_NAME": table_name,
        "RECORD_COUNT": len(fact),
        "DUPLICATE_GRAIN_KEYS": duplicate_count,
        "ORPHAN_RECORDS": orphan_count,
    })
    for column_name, null_count in fact.isnull().sum().items():
        if int(null_count) > 0:
            missingness.append({
                "TABLE_NAME": table_name,
                "COLUMN_NAME": column_name,
                "NULL_COUNT": int(null_count),
            })
display(pd.DataFrame(fact_checks))
display(pd.DataFrame(missingness))

## 2. Is the history long and coherent enough?

The earliest observation needs a full 365-day lookback. Daily servicing state supplies balances, limits, amount due, and delinquency ingredients; it does not store utilisation or rolling model features.

In [ ]:
daily = pd.read_snowflake(f"{database_name}.{raw_schema}.FACT_ACCOUNT_DAILY_SNAPSHOT")
financials = pd.read_snowflake(f"{database_name}.{raw_schema}.FACT_CUSTOMER_FINANCIAL_SNAPSHOT")

history_coverage = pd.DataFrame({
    "SOURCE": ["daily servicing", "monthly financial"],
    "FIRST_DATE": [daily["SNAPSHOT_DATE"].min(), financials["SNAPSHOT_DATE"].min()],
    "LAST_DATE": [daily["SNAPSHOT_DATE"].max(), financials["SNAPSHOT_DATE"].max()],
    "DISTINCT_DATES": [daily["SNAPSHOT_DATE"].nunique(), financials["SNAPSHOT_DATE"].nunique()],
})
display(history_coverage)

daily_summary = daily[[
    "OUTSTANDING_BALANCE", "CREDIT_LIMIT", "AVAILABLE_CREDIT",
    "AMOUNT_DUE", "MINIMUM_PAYMENT_DUE", "DAYS_PAST_DUE", "ARREARS_AMOUNT",
]].describe()
display(daily_summary)

Aggregates can hide whether the simulated behaviour is longitudinal. Inspect a small sample of facilities through time and derive utilisation only for this analysis.

In [ ]:
default_events = pd.read_snowflake(f"{database_name}.{raw_schema}.FACT_DEFAULT_EVENT")
sample_default_accounts = default_events["ACCOUNT_ID"].head(2).tolist()
sample_current_accounts = accounts[~accounts["ACCOUNT_ID"].isin(sample_default_accounts)]["ACCOUNT_ID"].head(2).tolist()
sample_accounts = sample_default_accounts + sample_current_accounts

trajectories = daily[daily["ACCOUNT_ID"].isin(sample_accounts)].copy()
trajectories["UTILISATION"] = trajectories["OUTSTANDING_BALANCE"] / trajectories["CREDIT_LIMIT"]
trajectory_monthly = trajectories.groupby(["ACCOUNT_ID", pd.Grouper(key="SNAPSHOT_DATE", freq="MS")]).agg({
    "UTILISATION": "mean", "DAYS_PAST_DUE": "max", "CREDIT_LIMIT": "max",
}).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for account_id in sample_accounts:
    account_history = trajectory_monthly[trajectory_monthly["ACCOUNT_ID"] == account_id]
    axes[0].plot(account_history["SNAPSHOT_DATE"], account_history["UTILISATION"], label=account_id)
    axes[1].plot(account_history["SNAPSHOT_DATE"], account_history["DAYS_PAST_DUE"], label=account_id)
axes[0].set_ylabel("Mean utilisation")
axes[1].set_ylabel("Maximum days past due")
axes[1].set_xlabel("Month")
axes[0].legend(ncol=2)
plt.tight_layout()

## 3. What do the atomic events add?

Payments describe realised repayment behaviour. Contacts and account events describe servicing pressure and limit changes. Their timestamps make trailing windows possible without leaking future events.

In [ ]:
payments = pd.read_snowflake(f"{database_name}.{raw_schema}.FACT_PAYMENT")
contacts = pd.read_snowflake(f"{database_name}.{raw_schema}.FACT_CUSTOMER_CONTACT")
account_events = pd.read_snowflake(f"{database_name}.{raw_schema}.FACT_ACCOUNT_EVENT")

payment_profile = payments.groupby("PAYMENT_STATUS").agg({
    "PAYMENT_ID": "count", "PAYMENT_AMOUNT": ["mean", "median"],
}).reset_index()
contact_profile = contacts.groupby(["CONTACT_REASON", "CONTACT_OUTCOME"]).size().reset_index(name="CONTACT_COUNT")
event_profile = account_events.groupby("EVENT_TYPE").size().reset_index(name="EVENT_COUNT")
display(payment_profile)
display(contact_profile)
display(event_profile)

In [ ]:
sample_payment_history = payments[payments["ACCOUNT_ID"].isin(sample_accounts)][[
    "ACCOUNT_ID", "PAYMENT_DATE", "PAYMENT_AMOUNT", "PAYMENT_STATUS",
]].sort_values(["ACCOUNT_ID", "PAYMENT_DATE"])
sample_contact_history = contacts[contacts["ACCOUNT_ID"].isin(sample_accounts)][[
    "ACCOUNT_ID", "CONTACT_TIMESTAMP", "CONTACT_REASON", "CONTACT_OUTCOME",
]].sort_values(["ACCOUNT_ID", "CONTACT_TIMESTAMP"])
display(sample_payment_history.tail(24))
display(sample_contact_history.tail(24))

## 4. When does the target become knowable?

Only now introduce the separate observation and outcome relation. Inspect label finality and observation coverage, but do not inspect target rates in the controlled-drift hold-out period.

In [ ]:
observations = pd.read_snowflake(f"{database_name}.{raw_schema}.ACCOUNT_OBSERVATION")
label_coverage = observations.groupby("OUTCOME_STATUS").agg({
    "ACCOUNT_ID": "count", "OBSERVATION_DATE": ["min", "max"],
    "OUTCOME_FINALITY_DATE": "max",
}).reset_index()
display(label_coverage)

holdout_cutoff = pd.Timestamp(config["data"]["drift_start_date"])
pre_holdout = observations[
    (observations["OUTCOME_STATUS"] == "FINALISED")
    & (observations["OBSERVATION_DATE"] < holdout_cutoff)
]
monthly_target = pre_holdout.groupby("OBSERVATION_DATE")["DEFAULT_WITHIN_90D"].agg(["count", "mean"]).reset_index()
monthly_target.columns = ["OBSERVATION_DATE", "OBSERVATION_COUNT", "DEFAULT_RATE"]
display(monthly_target)

The account-month observation grain matches monthly case prioritisation. A development and validation boundary still needs to be selected from visible pre-holdout coverage in notebook 02; it is not a project-wide configuration decision.

## 5. Provisional feature hypotheses

**Account profile:** age, customer tenure, origination channel, current limit and income, limit-to-income, and limit-change recency. These change slowly and describe capacity or relationship context.

**Balance and utilisation:** current balance, current utilisation, trailing 30/90-day utilisation, volatility, balance changes, days over limit, and available-credit ratio. These describe how intensively the facility is used.

**Payment behaviour:** 30/90-day payment sums, payment counts, payment-to-balance and payment-to-due ratios, missed-payment counts, recency, and trend. These describe whether repayment behaviour is weakening.

**Delinquency and contact:** current and trailing maximum days past due, delinquent days, episodes, consecutive delinquency, recency, contacts, and broken promises. These describe servicing stress.

All windows are backward-looking and end at the observation timestamp. `DATA_SCENARIO`, generated timestamps, identifiers, outcome finality, and default events are controls or labels, not predictors. Notebook 02 must develop and inspect each family before registering it.